# Mejorando modelo

### Busqueda solo de Epochs y Batch Size

In [ ]:
from nltk.translate.bleu_score import corpus_bleu
from tqdm import tqdm
from tensorflow.keras.models import load_model

# Lista de modelos y combinaciones de hiperparámetros
model_configs = [
    {"epochs": 10, "batch_size": 32},
    {"epochs": 10, "batch_size": 64},
    {"epochs": 20, "batch_size": 64},
    {"epochs": 30, "batch_size": 64},
]

# Guardar resultados
bleu_results = []

# Evaluar cada modelo
for config in model_configs:
    model_name = f"caption_model_e{config['epochs']}_b{config['batch_size']}.h5"
    print(f"Evaluando modelo: {model_name}")
    model = model

    test_keys = list(features.keys())[:1000]  # Subconjunto
    actual, predicted = list(), list()

    for key in tqdm(test_keys, desc="Generando captions"):
        references = [caption.split() for caption in descriptions[key]]
        photo = features[key].reshape((1, 4096))
        y_pred = generate_caption(model, tokenizer, photo, max_len).split()

        actual.append(references)
        predicted.append(y_pred)

    # Calcular BLEU Scores
    bleu1 = corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0))
    bleu2 = corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0))
    bleu3 = corpus_bleu(actual, predicted, weights=(0.33, 0.33, 0.33, 0))
    bleu4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25))

    bleu_results.append({
        "model_name": model_name,
        "epochs": config["epochs"],
        "batch_size": config["batch_size"],
        "bleu1": bleu1,
        "bleu2": bleu2,
        "bleu3": bleu3,
        "bleu4": bleu4
    })

# Encontrar el mejor modelo (por BLEU-4)
best_model = max(bleu_results, key=lambda x: x["bleu4"])

# Imprimir resultados del mejor modelo
print("\n🟢 Mejor modelo encontrado:")
print(f"Modelo: {best_model['model_name']}")
print(f"Epochs: {best_model['epochs']}, Batch Size: {best_model['batch_size']}")
print(f"BLEU-1: {best_model['bleu1']:.6f}")
print(f"BLEU-2: {best_model['bleu2']:.6f}")
print(f"BLEU-3: {best_model['bleu3']:.6f}")
print(f"BLEU-4: {best_model['bleu4']:.6f}")


Evaluando modelo: caption_model_e10_b32.h5


Generando captions: 100%|██████████| 1000/1000 [06:56<00:00,  2.40it/s]


Evaluando modelo: caption_model_e10_b64.h5


Generando captions: 100%|██████████| 1000/1000 [07:01<00:00,  2.37it/s]


Evaluando modelo: caption_model_e20_b64.h5


Generando captions: 100%|██████████| 1000/1000 [07:15<00:00,  2.30it/s]


Evaluando modelo: caption_model_e30_b64.h5


Generando captions: 100%|██████████| 1000/1000 [07:21<00:00,  2.27it/s]



🟢 Mejor modelo encontrado:
Modelo: caption_model_e10_b32.h5
Epochs: 10, Batch Size: 32
BLEU-1: 0.454463
BLEU-2: 0.317014
BLEU-3: 0.237320
BLEU-4: 0.181460


### Busqueda hiperparametros

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.optimizers import Adam

def create_model(vocab_size, max_len, learning_rate):
    # Imagen
    inputs1 = Input(shape=(4096,))
    fe1 = Dropout(0.4)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    # Texto
    inputs2 = Input(shape=(max_len,))
    se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
    se2 = Dropout(0.4)(se1)
    se3 = LSTM(256)(se2)

    # Fusionar
    decoder1 = add([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)

    model = Model(inputs=[inputs1, inputs2], outputs=outputs)

    # Compilar con optimizador ajustable
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(loss='categorical_crossentropy', optimizer=optimizer)

    return model


In [ ]:
import time
import itertools
import os  # Asegúrate de importar os

# Define los valores que quieres probar
epoch_options = [30]
batch_size_options = [64]
learning_rate_options = [1e-3]

# Todas las combinaciones posibles
param_grid = list(itertools.product(epoch_options, batch_size_options, learning_rate_options))

for idx, (epochs, batch_size, learning_rate) in enumerate(param_grid):
    model_name = f"caption_model_e{epochs}_b{batch_size}_lr{learning_rate:.0e}.h5"
    
    if os.path.exists(model_name):
        print(f"\n⏩ Modelo ya existe: {model_name}, saltando...")
        continue

    print(f"\n🔁 Combinación {idx+1}/{len(param_grid)}")
    print(f"Epochs: {epochs}, Batch Size: {batch_size}, Learning Rate: {learning_rate}")

    # Crear modelo con hiperparámetros actuales
    model = create_model(vocab_size, max_len, learning_rate)

    steps = sum(len(c) for c in descriptions.values()) // batch_size
    start_time = time.time()

    for epoch in range(epochs):
        print(f"  Epoch {epoch+1}/{epochs}")
        generator = data_generator(descriptions, features, tokenizer, max_len, batch_size)
        model.fit(generator, epochs=1, steps_per_epoch=steps, verbose=1)

    # Guardar el modelo entrenado
    model.save(model_name)

    elapsed_time = time.time() - start_time
    print(f"✅ Guardado: {model_name}")   
    print(f"⏱️ Tiempo total: {elapsed_time:.2f} segundos")
    print("-----------------------------------------------------")



🔁 Combinación 1/1
Epochs: 30, Batch Size: 64, Learning Rate: 0.001
  Epoch 1/30
632/632 [==============================] - 280s 438ms/step - loss: 5.6813
  Epoch 2/30
632/632 [==============================] - 295s 467ms/step - loss: 4.7048
  Epoch 3/30
632/632 [==============================] - 330s 522ms/step - loss: 4.2467
  Epoch 4/30
632/632 [==============================] - 319s 505ms/step - loss: 3.9453
  Epoch 5/30
632/632 [==============================] - 329s 520ms/step - loss: 3.7002
  Epoch 6/30
632/632 [==============================] - 330s 522ms/step - loss: 3.4975
  Epoch 7/30
632/632 [==============================] - 326s 516ms/step - loss: 3.2986
  Epoch 8/30
632/632 [==============================] - 332s 525ms/step - loss: 3.1318
  Epoch 9/30
632/632 [==============================] - 334s 528ms/step - loss: 2.9737
  Epoch 10/30
632/632 [==============================] - 335s 530ms/step - loss: 2.8282
  Epoch 11/30
632/632 [==============================] - 338s

c:\Users\WD\.conda\envs\tf_gpu\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


✅ Guardado: caption_model_e30_b64_lr1e-03.h5
⏱️ Tiempo total: 10407.78 segundos
-----------------------------------------------------


In [ ]:
model_hiper = load_model(".h5")

# Verificar el resumen del modelo
model_hiper.summary()

### Probar Modelos

In [ ]:
import os
from tensorflow.keras.models import load_model
from nltk.translate.bleu_score import corpus_bleu
from tqdm import tqdm

# Parámetros usados en el entrenamiento
epoch_options = [30]
batch_size_options = [64]
learning_rate_options = [1e-3, 5e-5]

# Ruta de los modelos (ajusta si es necesario)
model_dir = "./"

# Lista para guardar resultados
results = []

# Iterar sobre todas las combinaciones posibles
for epochs in epoch_options:
    for batch_size in batch_size_options:
        for lr in learning_rate_options:
            # Construir nombre de archivo
            model_filename = f"caption_model_e{epochs}_b{batch_size}_lr{lr:.0e}.h5"
            model_path = os.path.join(model_dir, model_filename)

            # Verificar si el archivo existe
            if os.path.exists(model_path):
                print(f"\n📂 Evaluando modelo: {model_filename}")
                model = load_model(model_path)

                # Evaluación del BLEU Score
                actual, predicted = [], []
                test_keys = list(features.keys())[:1000]

                for key in tqdm(test_keys):
                    # Referencias reales
                    references = [d.split() for d in descriptions[key]]
                    photo = features[key].reshape((1, 4096))
                    y_pred = generate_caption(model, tokenizer, photo, max_len).split()
                    actual.append(references)
                    predicted.append(y_pred)

                # Calcular BLEU
                bleu1 = corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0))
                bleu2 = corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0))
                bleu3 = corpus_bleu(actual, predicted, weights=(0.33, 0.33, 0.33, 0))
                bleu4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25))

                # Guardar resultados
                results.append({
                    'model': model_filename,
                    'epochs': epochs,
                    'batch_size': batch_size,
                    'learning_rate': lr,
                    'BLEU-1': bleu1,
                    'BLEU-2': bleu2,
                    'BLEU-3': bleu3,
                    'BLEU-4': bleu4
                })

                print(f"✅ BLEU Scores:")
                print(f"   BLEU-1: {bleu1:.4f}")
                print(f"   BLEU-2: {bleu2:.4f}")
                print(f"   BLEU-3: {bleu3:.4f}")
                print(f"   BLEU-4: {bleu4:.4f}")
            else:
                print(f"❌ Modelo no encontrado: {model_filename}")

# Mostrar resumen ordenado por BLEU-4 descendente
results = sorted(results, key=lambda x: x['BLEU-4'], reverse=True)
print("\n📊 Resultados ordenados por BLEU-4:")
for res in results:
    print(f"{res['model']}")
    print(f"   BLEU-1: {res['BLEU-1']:.4f}")
    print(f"   BLEU-2: {res['BLEU-2']:.4f}")
    print(f"   BLEU-3: {res['BLEU-3']:.4f}")
    print(f"   BLEU-4: {res['BLEU-4']:.4f}\n")



📂 Evaluando modelo: caption_model_e30_b64_lr1e-03.h5


100%|██████████| 1000/1000 [09:51<00:00,  1.69it/s]


✅ BLEU Scores:
   BLEU-1: 0.4111
   BLEU-2: 0.2695
   BLEU-3: 0.1859
   BLEU-4: 0.1292
❌ Modelo no encontrado: caption_model_e30_b64_lr5e-05.h5

📊 Resultados ordenados por BLEU-4:
caption_model_e30_b64_lr1e-03.h5
   BLEU-1: 0.4111
   BLEU-2: 0.2695
   BLEU-3: 0.1859
   BLEU-4: 0.1292

